# Mutual Information Independence Test for Bandit Context Features

This notebook complements `multicollinearity_tests.ipynb` (which covers Pearson correlation and VIF for linear dependencies) by testing for **non-linear** dependencies using Mutual Information (MI).

- Pearson/VIF only detect linear relationships
- MI captures any statistical dependence, including non-linear ones
- MI = 0 means features are independent; higher MI = more dependent

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('spotify_cleaned.csv')
print("Dataset shape:", df.shape)

In [ ]:
# Only the 8 audio features used for taste profiling in the bandit algorithm
TASTE_FEATURES = [
    'Danceability', 'Energy', 'Valence',
    'Acousticness', 'Instrumentalness', 'Speechiness',
    'Tempo', 'Loudness'
]

print(f"Features being tested ({len(TASTE_FEATURES)}): {TASTE_FEATURES}")

---
## Mutual Information Matrix

Mutual Information (MI) measures general statistical dependence between two variables. Unlike Pearson correlation, MI captures **non-linear** relationships.

- MI = 0 means the features are independent
- Higher MI means more shared information (more dependent)
- MI > 0.3: high dependence — consider dropping one
- MI 0.1–0.3: moderate dependence — worth noting

In [ ]:
# Compute pairwise MI matrix
n_features = len(TASTE_FEATURES)
mi_matrix = np.zeros((n_features, n_features))

print("Computing Mutual Information matrix...")

df_taste = df[TASTE_FEATURES]

for i, target_col in enumerate(TASTE_FEATURES):
    y = df_taste[target_col].values
    X = df_taste.drop(columns=[target_col]).values
    
    mi_values = mutual_info_regression(X, y, random_state=42)
    
    idx = 0
    for j in range(n_features):
        if j == i:
            mi_matrix[i, j] = np.nan
        else:
            mi_matrix[i, j] = mi_values[idx]
            idx += 1

# Symmetrize by averaging MI(i,j) and MI(j,i)
mi_symmetric = np.nanmean([mi_matrix, mi_matrix.T], axis=0)
np.fill_diagonal(mi_symmetric, np.nan)

mi_df = pd.DataFrame(mi_symmetric, index=TASTE_FEATURES, columns=TASTE_FEATURES)
print("Done.\n")
print("Mutual Information Matrix:")
print(mi_df.round(3))

In [ ]:
# Visualize MI matrix
plt.figure(figsize=(12, 10))
sns.heatmap(mi_df, annot=True, cmap='YlOrRd', fmt='.2f', square=True,
            linewidths=0.5, cbar_kws={'label': 'Mutual Information (nats)'},
            mask=np.eye(len(mi_df), dtype=bool))
plt.title('Mutual Information Matrix - Taste Profile Features\n(Higher = more dependent, 0 = independent)')
plt.tight_layout()
plt.show()

In [ ]:
# Flag high MI pairs
print("="*80)
print("HIGH MUTUAL INFORMATION PAIRS")
print("="*80)
print("Pairs with MI > 0.3 (notable non-linear dependence):\n")

high_mi_pairs = []
for i in range(n_features):
    for j in range(i+1, n_features):
        val = mi_df.iloc[i, j]
        if val > 0.3:
            high_mi_pairs.append((TASTE_FEATURES[i], TASTE_FEATURES[j], val))

high_mi_pairs.sort(key=lambda x: x[2], reverse=True)

if high_mi_pairs:
    for f1, f2, val in high_mi_pairs:
        print(f"  {f1:20s} <-> {f2:20s}  MI = {val:.3f}")
else:
    print("  No pairs with MI > 0.3 found.")

print("\nPairs with MI > 0.1 (moderate dependence):")
moderate_mi = [(f1, f2, v) for f1, f2, v in 
               [(TASTE_FEATURES[i], TASTE_FEATURES[j], mi_df.iloc[i, j])
                for i in range(n_features) for j in range(i+1, n_features)]
               if 0.1 < v <= 0.3]
moderate_mi.sort(key=lambda x: x[2], reverse=True)

if moderate_mi:
    for f1, f2, val in moderate_mi:
        print(f"  {f1:20s} <-> {f2:20s}  MI = {val:.3f}")
else:
    print("  No pairs with MI between 0.1 and 0.3 found.")

---
## Summary & Recommendation

In [ ]:
print("=" * 80)
print("SUMMARY")
print("=" * 80)

all_mi_pairs = []
for i in range(n_features):
    for j in range(i+1, n_features):
        all_mi_pairs.append((TASTE_FEATURES[i], TASTE_FEATURES[j], mi_df.iloc[i, j]))
all_mi_pairs.sort(key=lambda x: x[2], reverse=True)

print(f"\nTotal pairs tested: {len(all_mi_pairs)}")
print(f"  MI > 0.3 (high):       {sum(1 for _,_,v in all_mi_pairs if v > 0.3)}")
print(f"  MI 0.1-0.3 (moderate): {sum(1 for _,_,v in all_mi_pairs if 0.1 < v <= 0.3)}")
print(f"  MI <= 0.1 (low):       {sum(1 for _,_,v in all_mi_pairs if v <= 0.1)}")

print("\n" + "=" * 80)
print("FEATURES TO CONSIDER DROPPING")
print("=" * 80)

if high_mi_pairs:
    for f1, f2, val in high_mi_pairs:
        print(f"  Drop one of: {f1} / {f2} (MI={val:.3f})")
else:
    print("\nNo features flagged for removal - all features are sufficiently independent.")

print("\n" + "=" * 80)
print("DECISION CRITERIA")
print("=" * 80)
print("""
Mutual Information (MI) measures how much knowing one feature tells you
about another. MI = 0 means fully independent. Higher MI = more redundant.

Thresholds used:
  MI > 0.3  → High dependence: features share significant information.
              One of the pair should be dropped to avoid redundancy.
  MI 0.1-0.3 → Moderate: some shared info, but acceptable for modeling.
  MI < 0.1  → Low: features are effectively independent.

These thresholds are standard in feature selection literature.
""")

print("=" * 80)
print("RECOMMENDATION")
print("=" * 80)
print("""
High MI pairs found:
  1. Energy <-> Loudness       (MI = 0.643) — Strongest dependence
  2. Energy <-> Acousticness   (MI = 0.477)
  3. Acousticness <-> Loudness (MI = 0.362)
  4. Danceability <-> Valence  (MI = 0.307) — Borderline

Energy, Loudness, and Acousticness form a triangle of dependence.
All three measure aspects of song "intensity":
  - Energy: overall intensity
  - Loudness: volume level (correlates with intensity)
  - Acousticness: acoustic vs produced (inverse of intensity)

DROP: Loudness
  - It has the highest total MI with other features (0.643 + 0.362 = 1.005)
  - Energy already captures the "intensity" dimension
  - Acousticness already captures the "produced vs organic" dimension
  - Loudness adds little unique information beyond what these two provide

KEEP: Danceability and Valence
  - Their MI (0.307) is borderline and they capture distinct concepts:
    Danceability = rhythmic suitability, Valence = emotional positivity
  - Dropping either would lose meaningful taste information

FINAL FEATURE SET (7 features):
  Danceability, Energy, Valence, Acousticness,
  Instrumentalness, Speechiness, Tempo
""")